# Grover's Search Algorithm

## Given a phase oracle for a predicate with $M$ marked items out of $N = 2^n$, Grover finds a marked item using only $O(\sqrt{N/M})$ oracle calls — a **quadratic** speedup over the $\Theta(N)$ required classically.

# 1. Set-up

## Let $f : \{0,1\}^n \to \{0,1\}$ mark a subset of size $M$. We are given the phase oracle

$$ \Large  O_f : |x\rangle \mapsto (-1)^{f(x)}|x\rangle. $$

## Define $|\psi\rangle = H^{\otimes n}|0\rangle = \frac{1}{\sqrt N}\sum_x |x\rangle$, $|s\rangle = \frac{1}{\sqrt M}\sum_{x : f(x)=1} |x\rangle$ (the "good" subspace), and $|s^\perp\rangle$ (the "bad" subspace orthogonal to $|s\rangle$).

## Then $|\psi\rangle = \sin\theta\,|s\rangle + \cos\theta\,|s^\perp\rangle$ where $\sin\theta = \sqrt{M/N}$.

# 2. The Grover iterate

## One iteration of Grover is

$$ \Large  G = D \cdot O_f, \qquad D = 2|\psi\rangle\langle\psi| - I. $$

## $D$ is the **diffusion operator** — it inverts amplitudes around their mean. Geometrically, $O_f$ reflects $|\psi\rangle$ across $|s^\perp\rangle$, and $D$ reflects across $|\psi\rangle$. The composition is a **rotation in the two-dimensional plane spanned by $|s\rangle$ and $|s^\perp\rangle$ by angle $2\theta$ toward $|s\rangle$**.

# 3. Optimal number of iterations

## After $k$ iterations the state is

$$ \Large  G^k |\psi\rangle = \sin((2k+1)\theta)\, |s\rangle + \cos((2k+1)\theta)\, |s^\perp\rangle. $$

## Measure to find a marked item with probability $\sin^2((2k+1)\theta)$. This is maximised when $(2k+1)\theta \approx \pi/2$, giving

$$ \Large  k^\star = \Big\lfloor \frac{\pi}{4}\sqrt{N/M} \Big\rfloor. $$

## Hence $O(\sqrt{N/M})$ oracle calls.

# 4. The diffusion operator as a circuit

## $D = H^{\otimes n} \,(2|0^n\rangle\langle 0^n| - I)\, H^{\otimes n}$. So the implementation is:

## 1. $H^{\otimes n}$
## 2. $X^{\otimes n}$
## 3. Multi-controlled $Z$ (flips the sign of $|1^n\rangle$)
## 4. $X^{\otimes n}$
## 5. $H^{\otimes n}$

## Steps 2-4 implement $2|0^n\rangle\langle 0^n| - I$ up to a global phase. The multi-controlled $Z$ is the expensive part — decomposing it into single- and two-qubit gates is the source of most of Grover's gate cost.

In [1]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import numpy as np

def grover_circuit(n, marked_states, iterations=None):
    N = 2**n
    M = len(marked_states)
    if iterations is None:
        iterations = int(np.floor((np.pi/4)*np.sqrt(N/M)))

    qc = QuantumCircuit(n, n)
    qc.h(range(n))                               # uniform superposition

    for _ in range(iterations):
        # phase oracle: flip phase of marked states
        for m in marked_states:
            bits = format(m, f'0{n}b')[::-1]     # little-endian
            for i, b in enumerate(bits):
                if b == '0':
                    qc.x(i)
            qc.h(n - 1)
            qc.mcx(list(range(n - 1)), n - 1)    # multi-controlled X = ... see below
            qc.h(n - 1)
            for i, b in enumerate(bits):
                if b == '0':
                    qc.x(i)
        # diffusion
        qc.h(range(n)); qc.x(range(n))
        qc.h(n - 1); qc.mcx(list(range(n - 1)), n - 1); qc.h(n - 1)
        qc.x(range(n)); qc.h(range(n))

    qc.measure(range(n), range(n))
    return qc, iterations

n = 4
marked = [11]                  # search for the basis state |1011>
qc, k = grover_circuit(n, marked)
print(f'Iterations: {k}')
sim = AerSimulator()
counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
print('Counts:', dict(sorted(counts.items(), key=lambda kv: -kv[1])[:5]))

Iterations: 3
Counts: {'1011': 985, '1100': 5, '0010': 5, '0000': 5, '1010': 4}


# 5. Why only quadratic?

## A famous lower bound (Bennett, Bernstein, Brassard, Vazirani 1997) says: any quantum algorithm solving unstructured search needs $\Omega(\sqrt{N})$ oracle calls. So Grover is **optimal up to a constant**.

## When additional structure is available (Walsh-Hadamard, algebraic, geometric, …), better algorithms exist. But for a truly opaque oracle, $\sqrt N$ is the wall.

# Recap

## - Each Grover iterate rotates the state by $2\theta \approx 2\sqrt{M/N}$ toward the marked subspace.
## - $\frac{\pi}{4}\sqrt{N/M}$ iterations maximises success probability.
## - Quadratic speedup is **optimal** for unstructured search.

## Next: amplitude amplification — the general framework.